## Creating an a summary agent for control performance.
### Approach to be used
- Create an endpoint for a table that will store the control summary
- An agent that will retrive control exceptions and relevant context using RAG.
- An agent should review those exceptions and record an insightful summary.
- An agent should review that summary and determine if it sufficient to be recorded.

Firstly, create and endpoint to be used to store agent feedback

Import all necessary libraries

In [1]:
#import os
from dotenv import load_dotenv
from agents import Agent, Runner,trace, function_tool
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
import requests
import asyncio
import httpx
from typing import Any
import os
#import json
load_dotenv(override=True)

True

Create a client for the anthropic's API

In [2]:
anthropic_client = AsyncOpenAI(
    api_key= os.getenv(key='ANTHROPIC_API_KEY'),
    base_url='https://api.anthropic.com/v1/',
)

Create the model

In [3]:
anthropic_model = OpenAIChatCompletionsModel(
            model="claude-haiku-4-5",
            openai_client=anthropic_client,
            )

retrieve all the required data

In [ ]:
'/data/exception'

sychronous approach

In [12]:
base_URL = 'https://controlweb-supabase.azurewebsites.net'
end_point_list = ['/data/exception','/data/logic','/data/dictionary']  #This takes 27 seconds 
results = {}

for end_point in end_point_list:
    response = requests.get(base_URL+end_point)
    if response.status_code == 200:
        results[end_point.split('/')[-1]] = response.json()
    

In [13]:
print(results)

{'exception': [{'name': 'Charles Hernandez', 'account_number': 'ACC010', 'registration_date': '2019-11-20', 'phone': '+12345678909', 'status': 'Suspended', 'usage_amount': 90.0, 'email': 'charles.hernandez@example.com', 'timestamp': '2023-09-01 08:25:00', 'user_id': 'USR010', 'detection_time': '2026-04-02T16:40:09.087326'}, {'name': 'Isaac Harris', 'account_number': 'ACC042', 'registration_date': '2021-09-12', 'phone': '+12345678941', 'status': 'Suspended', 'usage_amount': 200.0, 'email': 'isaac.harris@example.com', 'timestamp': '2023-09-01 17:50:00', 'user_id': 'USR042', 'detection_time': '2026-04-02T16:40:09.087326'}], 'logic': [{'control_logic': "SELECT *\n                    FROM raw_control_datalake.dev.raw_synthetic_data\n                    WHERE status = 'Suspended';\n                    ", 'created_timestamp': '2026-04-08T10:20:42.380718', 'reference_number': 1, 'control_logic_description': 'Check suspended customers', 'control_logic_status': 'ACTIVE'}], 'dictionary': [{'field

Asychronous Approach

In [14]:


base_URL = 'https://controlweb-supabase.azurewebsites.net'
end_point_list = ['/data/exception', '/data/logic', '/data/dictionary'] # This takes 1.2 seconds
async def fetch(client, end_point):
    response = await client.get(base_URL + end_point)
    if response.status_code == 200:
        return end_point.split("/")[-1], response.json()
    return end_point.end_point.split("/")[-1], None

async def fetch_all():
    async with httpx.AsyncClient() as client:
        tasks = [fetch(client, ep) for ep in end_point_list]
        responses = await asyncio.gather(*tasks)
        return {ep: data for ep, data in responses if data is not None}
results = await fetch_all()  

In [15]:
print(results)

{'exception': [{'name': 'Charles Hernandez', 'account_number': 'ACC010', 'registration_date': '2019-11-20', 'phone': '+12345678909', 'status': 'Suspended', 'usage_amount': 90.0, 'email': 'charles.hernandez@example.com', 'timestamp': '2023-09-01 08:25:00', 'user_id': 'USR010', 'detection_time': '2026-04-02T16:40:09.087326'}, {'name': 'Isaac Harris', 'account_number': 'ACC042', 'registration_date': '2021-09-12', 'phone': '+12345678941', 'status': 'Suspended', 'usage_amount': 200.0, 'email': 'isaac.harris@example.com', 'timestamp': '2023-09-01 17:50:00', 'user_id': 'USR042', 'detection_time': '2026-04-02T16:40:09.087326'}], 'logic': [{'control_logic': "SELECT *\n                    FROM raw_control_datalake.dev.raw_synthetic_data\n                    WHERE status = 'Suspended';\n                    ", 'created_timestamp': '2026-04-08T10:20:42.380718', 'reference_number': 1, 'control_logic_description': 'Check suspended customers', 'control_logic_status': 'ACTIVE'}], 'dictionary': [{'field

Create an agent tool 

In [4]:


ALL_ENDPOINTS = ["/data/exception", "/data/logic", "/data/dictionary"]


async def _fetch_one(client: httpx.AsyncClient, endpoint: str) -> tuple[str, Any]:
    BASE_URL = "https://controlweb-supabase.azurewebsites.net"
    key = endpoint.split("/")[-1]
    try:
        response = await client.get(BASE_URL + endpoint)
        if response.status_code == 200:
            return key, response.json()
    except httpx.RequestError:
        pass
    return key, None

@function_tool
async def fetch_controlweb_data(endpoints: list[str] | None = None) -> dict[str, Any]:
    """
    Fetch ControlWeb data from one or more endpoints concurrently.

    Args:
        endpoints: Subset of ['/data/exception', '/data/logic', '/data/dictionary'].
                   Defaults to all three if not provided.

    Returns:
        Dict keyed by endpoint name ('exception', 'logic', 'dictionary').
        Keys for failed/non-200 requests are omitted.
    """
    targets = endpoints if endpoints is not None else ALL_ENDPOINTS
    async with httpx.AsyncClient() as client:
        tasks = [_fetch_one(client, ep) for ep in targets]
        results = await asyncio.gather(*tasks)
    return {key: data for key, data in results if data is not None}

In [17]:
print(fetch_controlweb_data)

FunctionTool(name='fetch_controlweb_data', description='Fetch ControlWeb data from one or more endpoints concurrently.', params_json_schema={'properties': {'endpoints': {'anyOf': [{'items': {'type': 'string'}, 'type': 'array'}, {'type': 'null'}], 'description': "Subset of ['/data/exception', '/data/logic', '/data/dictionary'].\n       Defaults to all three if not provided.", 'title': 'Endpoints'}}, 'title': 'fetch_controlweb_data_args', 'type': 'object', 'additionalProperties': False, 'required': ['endpoints']}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7d67185f0800>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)


In [5]:
system_prompt = f"""You are a Fraud Analyst assistant specializing in the review of automated control exceptions.

## INSTRUCTIONS

You will be given a tool. Before doing any analysis, you MUST call the
`rewrite_data` tool to retrieve the data you need.

Do not proceed with analysis until you get feedback from the rewrite_data tool.

## DATA YOU ARE FETCHING

Each endpoint will return some combination of the following:

- **Data dictionary** — field definitions and value descriptions
- **Control information** — the control's name, description, and exception-generation logic
- **Exception list** — the records flagged by the control

## YOUR TASK

Once all data has been fetched, produce a structured summary covering:

1. **Control Overview**
   What the control is designed to detect and why it matters from a fraud risk perspective —
   in plain language.

2. **Exception Population**
   Volume, key patterns, and notable characteristics of the flagged records.

3. **Analytical Interpretation**
   How the exceptions relate to the control logic. Highlight anything unusual, unexpected,
   or high-priority.

4. **Data Quality Observations**
   Any limitations, gaps, or ambiguities in the data that may affect reliability or
   interpretation.

## GUIDELINES

- Always use the data dictionary to interpret field values accurately.
- Ground all observations strictly in the fetched data — do not speculate.
- Flag ambiguities where control logic or data is unclear.
- Be concise and actionable — prioritise information that helps a reviewer triage or escalate."""

### Creating an agent that will receive the data and convert to an easily data for the processing agent.

In [6]:
agent_1_system_prompt = """

You are an AI assistant that rewrites JSON files into easily processable output for downstream agents.

## INSTRUCTIONS

You will be given a list of endpoints. You MUST call the `fetch_controlweb_data` tool on every endpoint before doing anything else. Do not begin any analysis or rewriting until all fetch calls are complete.

## DATA STRUCTURE

Each endpoint returns some combination of:
- **Data dictionary** — field definitions and value descriptions
- **Control information** — control name, description, and exception-generation logic
- **Exception list** — records flagged by the control

## YOUR TASK

Rewrite the fetched data into a clear, structured format that another agent can easily read and summarise.

"""

In [7]:
Rewrite_data = Agent(name="AI assistant",
                      instructions=agent_1_system_prompt,
                      tools=[fetch_controlweb_data],
                      model="gpt-4o-mini"
                      )

creating user message for the receipent agent to convert data into readable information

In [8]:
message = f"Here is a list of endpoints :{ALL_ENDPOINTS} use it to extract the information to review"

Checking if the agent works properly

In [38]:
with trace("Extract the required data recent"):
    result = await Runner.run(Rewrite_data,message)

In [39]:
print(result)

RunResult:
- Last agent: Agent(name="AI assistant", ...)
- Final output (str):
    Here’s a structured summary of the fetched data from the specified endpoints:
    
    ### 1. Exception List
    This section includes records of flagged accounts.
    
    | Name               | Account Number | Registration Date | Phone         | Status    | Usage Amount | Email                       | Timestamp           | User ID | Detection Time                   |
    |--------------------|----------------|--------------------|---------------|-----------|---------------|-----------------------------|----------------------|---------|-----------------------------------|
    | Charles Hernandez   | ACC010         | 2019-11-20         | +12345678909  | Suspended | 90.0          | charles.hernandez@example.com | 2023-09-01 08:25:00 | USR010  | 2026-04-02T16:40:09.087326       |
    | Isaac Harris        | ACC042         | 2021-09-12         | +12345678941  | Suspended | 200.0         | isaac.harris@exam

Convert the rewrite agent to a tool

In [13]:
rewrite_tool = Rewrite_data.as_tool(tool_name="rewrite_tool", tool_description="an AI assistant that rewrites JSON files into easily processable output for downstream agents")

In [14]:
print(rewrite_tool)

FunctionTool(name='rewrite_tool', description='an AI assistant that rewrites JSON files into easily processable output for downstream agents', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7d83a82889b0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)


In [15]:
print(message)

Call the rewrite_data tool to extract the information to review


In [16]:
message = f"Call the rewrite_data tool to extract the information to review, provide the tool with following list of Endpoints: {ALL_ENDPOINTS}"

In [17]:
print(message)

Call the rewrite_data tool to extract the information to review, provide the tool with following list of Endpoints: ['/data/exception', '/data/logic', '/data/dictionary']


In [24]:
fraud_analyst = Agent(name="Fraud analyst",
                      instructions=system_prompt,
                      tools=[rewrite_tool],
                      model=anthropic_model,
                      )

In [25]:
with trace("Multi-agent outcome for reviewing data"):
    result = await Runner.run(fraud_analyst,message)

In [48]:
print(result.final_output)

### Control Overview
The control is designed to monitor and detect accounts that have been marked as "Suspended" due to potential fraudulent activity or policy violations. This is critical in mitigating fraud risk as suspended accounts may indicate misuse of the service or attempts to evade detection.

### Exception Population
- **Volume**: There are 2 flagged records in the exceptions list.
- **Key Patterns**:
  - Both records belong to individuals with unique identifiers (User IDs) and are marked as "Suspended."
  - The usage amounts of $90.0 and $200.0 indicate varying levels of account activity.
- **Notable Characteristics**: 
  - The accounts were registered over different periods, with the oldest being from 2019 and the recent one from 2021.

### Analytical Interpretation
- The control logic specifically looks for accounts that are in a suspended status, which in these cases they are.
- Notably, no unusual patterns were detected in the amounts; however, it’s noteworthy that both 

Create an agent that will review the output the provided by the processing agent 

In [23]:
print(result.final_output)

### Structured Summary of Control Exceptions

#### 1. Control Overview
The control is designed to identify accounts that have been suspended, which may indicate potential fraudulent activities. Monitoring suspended accounts is crucial because they represent users who may have attempted to misuse services or violate terms of use, thereby posing a significant fraud risk to the organization.

#### 2. Exception Population
- **Volume of Exceptions:** Two records have been flagged for suspension.
- **Key Patterns:**
  - Both flagged users maintain a unique identifier (User ID) and have been suspended for their accounts.
  - The flagged usage amounts show varying values ($90.00 and $200.00), suggesting different levels of engagement or possible misuse.
- **Notable Characteristics:**
  - Accounts like Charles Hernandez and Isaac Harris have distinct registration dates and usage amounts, which may signal different patterns of account behavior leading to suspension.

#### 3. Analytical Interpret

In [26]:
print(result.final_output)

Now I have the fetched data. Let me provide a structured analytical summary:

---

## FRAUD ANALYST REVIEW: SUSPENDED CUSTOMERS CONTROL

### 1. CONTROL OVERVIEW

**Control Name:** Check Suspended Customers

**Purpose:** This control identifies all customer accounts with a current status of "Suspended." It functions as a baseline monitoring mechanism to flag accounts that have been flagged as suspended, which may indicate:
- Compliance violations or risk remediation
- Potential account compromise or fraudulent activity
- Customer delinquency or contractual breaches
- Regulatory action or restrictions

**Why It Matters:** Suspended accounts represent heightened risk exposure. Monitoring them helps detect dormant fraud activity, account takeover persistence, or regulatory-flagged entities that should not be conducting transactions.

---

### 2. EXCEPTION POPULATION

**Volume:** 2 flagged records

**Key Characteristics:**

| Characteristic | Finding |
|---|---|
| **Account Status** | Both 